In [2]:
import sys
from pathlib import Path

current_dir = Path.cwd()
if (current_dir / "src").exists():
    sys.path.insert(0, str(current_dir / "src"))
elif (current_dir.parent / "src").exists():
    sys.path.insert(0, str(current_dir.parent / "src"))
elif current_dir.name == "src":
    sys.path.insert(0, str(current_dir))

import matplotlib.pyplot as plt
from spacerace_dqn.config3 import Config3
from spacerace_dqn.trainer3 import DQNTrainer3

# 4 experiments for Task 3
experiments = {
    "Epsilon (Linear)": Config3(
        exploration="epsilon", decay_schedule="linear", submission_name="t3_eps_lin"
    ),
    "Epsilon (Exponential)": Config3(
        exploration="epsilon", decay_schedule="exponential", submission_name="t3_eps_exp"
    ),
    "Boltzmann (Linear)": Config3(
        exploration="boltzmann", decay_schedule="linear", submission_name="t3_boltz_lin"
    ),
    "Boltzmann (Exponential)": Config3(
        exploration="boltzmann", decay_schedule="exponential", submission_name="t3_boltz_exp"
    )
}

results = {}

# Runs the comparative analysis
for name, cfg in experiments.items():
    print(f"\n{'='*50}\nStarting Experiment: {name}\n{'='*50}")
    trainer = DQNTrainer3(cfg)
    trainer.train()
    
    # Stores history for plotting
    results[name] = {
        "history": trainer.history,
        "eval_history": trainer.eval_history
    }
    print(f"Finished {name}. Final Eval Mean: {trainer.eval_history[-1]['mean_score']:.2f}")

print("\nAll experiments complete! Ready for plotting.")


Starting Experiment: Epsilon (Linear)
[DQNTrainer3]  device=cuda  state=(3, 54, 39)  params=158,850
  buffer=Uniform cap=10,000  batch=64  warmup=1,000  target_update=100  exploration=epsilon (linear)  semantic_info=False
  [warmup]  6,000 transitions from 10 heuristic episodes (0.2 min)
  ep=0001  step=006600  buf=6,600  train_score=0.00  eval_mean=5.00  loss=0.0160  q=0.281  explore=0.880  0.4m
  ep=0020  step=018000  buf=10,000  train_score=1.00  eval_mean=13.00  loss=0.0184  q=3.490  explore=0.664  3.8m
  ep=0040  step=030000  buf=10,000  train_score=5.00  eval_mean=25.00  loss=0.0196  q=4.018  explore=0.436  6.2m
  ep=0060  step=042000  buf=10,000  train_score=14.00  eval_mean=25.00  loss=0.0199  q=5.332  explore=0.208  8.6m
  ep=0080  step=054000  buf=10,000  train_score=24.00  eval_mean=22.00  loss=0.0140  q=5.736  explore=0.050  11.1m
  ep=0100  step=066000  buf=10,000  train_score=25.00  eval_mean=27.00  loss=0.0102  q=5.723  explore=0.050  13.5m
  ep=0120  step=078000  buf=1

In [3]:
import numpy as np
import matplotlib.pyplot as plt

# Smoothing function to make RL curves readable
def smooth(scalars, weight=0.85):
    if not scalars: return []
    last = scalars[0]
    smoothed = []
    for point in scalars:
        smoothed_val = last * weight + (1 - weight) * point
        smoothed.append(smoothed_val)
        last = smoothed_val
    return smoothed

# Set up the 2x2 plot grid
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Task 3: Exploration Strategies Comparison', fontsize=18, fontweight='bold')
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'] # Blue, Orange, Green, Red

for idx, (name, data) in enumerate(results.items()):
    history = data["history"]
    eval_history = data["eval_history"]
    c = colors[idx]
    
    # Evaluation Score (Generalization / Final Performance)
    eval_eps = [x["episode"] for x in eval_history]
    eval_scores = [x["mean_score"] for x in eval_history]
    axes[0, 0].plot(eval_eps, eval_scores, label=name, color=c, marker='o', markersize=4, linewidth=2)
    
    # Training Score (Learning Efficiency)
    train_eps = [x["episode"] for x in history]
    train_scores = [x["score"] for x in history]
    smoothed_train = smooth(train_scores)
    axes[0, 1].plot(train_eps, smoothed_train, label=name, color=c, linewidth=2)
    axes[0, 1].plot(train_eps, train_scores, color=c, alpha=0.15) # Show raw data faintly in background
    
    # Q-Values (Stability / Overestimation)
    q_vals = [x["q_value"] for x in history if not np.isnan(x["q_value"])]
    q_eps = train_eps[-len(q_vals):] # Align episodes
    smoothed_q = smooth(q_vals, weight=0.9)
    axes[1, 0].plot(q_eps, smoothed_q, label=name, color=c, linewidth=2)
    
    # Exploration Parameter Decay (Behavior)
    explores = [x["explore_param"] for x in history]
    # Normalize temperature (usually 5 to 0) to a 1-0 scale so it fits on the same graph as Epsilon
    if "Boltzmann" in name:
        max_temp = max(explores) if max(explores) > 0 else 1
        explores = [x / max_temp for x in explores]
        name_label = f"{name} (Normalized)"
    else:
        name_label = name
    axes[1, 1].plot(train_eps, explores, label=name_label, color=c, linewidth=2)

# Format Subplot 1: Eval Score
axes[0, 0].set_title('Evaluation Mean Score (Codabench Proxy)', fontsize=14)
axes[0, 0].set_xlabel('Episode'); axes[0, 0].set_ylabel('Score')
axes[0, 0].grid(alpha=0.3); axes[0, 0].legend()

# Format Subplot 2: Train Score
axes[0, 1].set_title('Smoothed Training Score (Efficiency)', fontsize=14)
axes[0, 1].set_xlabel('Episode'); axes[0, 1].set_ylabel('Score')
axes[0, 1].grid(alpha=0.3); axes[0, 1].legend()

# Format Subplot 3: Q-Values
axes[1, 0].set_title('Smoothed Q-Values (Stability)', fontsize=14)
axes[1, 0].set_xlabel('Episode'); axes[1, 0].set_ylabel('Estimated Q-Value')
axes[1, 0].grid(alpha=0.3); axes[1, 0].legend()

# Format Subplot 4: Exploration Decay
axes[1, 1].set_title('Exploration Parameter Decay', fontsize=14)
axes[1, 1].set_xlabel('Episode'); axes[1, 1].set_ylabel('Value (Normalized)')
axes[1, 1].grid(alpha=0.3); axes[1, 1].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

C:\Users\tiago\AppData\Local\Temp\ipykernel_28628\3323049908.py:75: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


In [4]:
from spacerace_dqn.submission import SubmissionPackager

# SET WINNER HERE based on the graphs!
WINNING_EXPLORATION = "boltzmann"   # "epsilon" or "boltzmann"
WINNING_DECAY       = "exponential" # "linear" or "exponential"

print(f"Training FINAL model using {WINNING_EXPLORATION} ({WINNING_DECAY})...")

# Configure for maximum performance (800 episodes)
final_cfg = Config3(
    exploration=WINNING_EXPLORATION,
    decay_schedule=WINNING_DECAY,
    episodes=800,              # Back to 800 for maximum Codabench performance
    eval_every=50,             # Evaluate less often to speed up training
    submission_name="task3_final_submission"
)

# Train the absolute best agent
final_trainer = DQNTrainer3(final_cfg)
best_agent = final_trainer.train()
final_eval = final_trainer.final_eval()

print(f"\nFinal Training Complete! Eval Score: {final_eval['mean_score']:.2f}")

# Package for Codabench
packager = SubmissionPackager(final_cfg)

# We pass an empty list for baselines since this is just the agent submission
summary = packager.package(
    best_agent,
    state_shape=final_trainer.state_shape,
    history=final_trainer.history,
    eval_history=final_trainer.eval_history,
    baselines=[], 
    final_eval=final_eval,
    train_minutes=final_trainer.train_minutes,
)

print("-" * 50)
print(f"SUCCESS! Submission ZIP is ready at:")
print(summary['files']['zip'])
print("-" * 50)

Training FINAL model using boltzmann (exponential)...
[DQNTrainer3]  device=cuda  state=(3, 54, 39)  params=158,850
  buffer=Uniform cap=10,000  batch=64  warmup=1,000  target_update=100  exploration=boltzmann (exponential)  semantic_info=False
  [warmup]  6,000 transitions from 10 heuristic episodes (0.2 min)
  ep=0001  step=006600  buf=6,600  train_score=0.00  eval_mean=2.00  loss=0.0160  q=0.280  explore=3.333  0.3m
  ep=0050  step=036000  buf=10,000  train_score=0.00  eval_mean=17.00  loss=0.0066  q=3.409  explore=0.502  9.0m
  ep=0100  step=066000  buf=10,000  train_score=9.00  eval_mean=26.00  loss=0.0140  q=5.076  explore=0.200  17.3m
  ep=0150  step=096000  buf=10,000  train_score=14.00  eval_mean=28.00  loss=0.0122  q=5.811  explore=0.200  25.5m
  ep=0200  step=126000  buf=10,000  train_score=9.00  eval_mean=27.00  loss=0.0087  q=5.802  explore=0.200  44.1m
  ep=0250  step=156000  buf=10,000  train_score=13.00  eval_mean=29.00  loss=0.0066  q=5.857  explore=0.200  53.9m
  ep=0